Gaussian kernel SVM can perfectly separate the two classes

# 7

In [ ]:
#| echo: false

import pandas as pd
import numpy as np
from sklearn import svm
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

df = pd.read_csv("data/wine_dataset.csv")

X = df.drop("style", axis=1)
y = df["style"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.5, random_state=1
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Logistic Regression Accuracy: {round(accuracy,2)}")

Cs = [0.01, 0.1, 1, 10, 100]

Gammas = [0.01, 0.1, 1, 10]

Degrees = [2, 3, 4]

print(f"Testing with paramaters: Cs={Cs}, Gammas={Gammas}, Degrees={Degrees}\n")

best_lin_acc = 0
best_lin_params = {}
best_rbf_acc = 0
best_rbf_params = {}
best_poly_acc = 0
best_poly_params = {}
count = 0
for C in Cs:
  svm_lin = svm.SVC(kernel='linear', C=C, random_state=1)
  svm_lin.fit(X_train_scaled, y_train)
  y_pred = svm_lin.predict(X_test_scaled)
  lin_acc = accuracy_score(y_test, y_pred)
  if lin_acc > best_lin_acc:
    best_lin_acc = lin_acc
    best_lin_params = {'C': C}
  for gamma in Gammas:
    if gamma == 10 and C == 100:
      break
    svm_rbf = svm.SVC(kernel='rbf', C=C, gamma=gamma, random_state=1)
    svm_rbf.fit(X_train_scaled, y_train)
    y_pred = svm_rbf.predict(X_test_scaled)
    rbf_acc = accuracy_score(y_test, y_pred)
    if rbf_acc > best_rbf_acc:
      best_rbf_acc = rbf_acc
      best_rbf_params = {'C': C, 'gamma': gamma}
    for degree in Degrees: 
      svm_poly = svm.SVC(kernel='poly', C=C, gamma=gamma, degree=degree, random_state=1)
      svm_poly.fit(X_train_scaled, y_train)
      y_pred = svm_poly.predict(X_test_scaled)
      poly_acc = accuracy_score(y_test, y_pred)
      if poly_acc > best_poly_acc:
        best_poly_acc = poly_acc
        best_poly_params = {'C': C, 'gamma': gamma, 'degree': degree}

      count += 1
      print(f"Completed {count} / {len(Cs) * len(Gammas) * len(Degrees)} parameter combinations", end="\r")
print(f"Best Linear SVM test accuracy: {best_lin_acc} (params: {best_lin_params})\n")
print(f"Best Polynomial SVM test accuracy: {best_poly_acc} (params: {best_poly_params})\n")
print(f"Best RBF SVM test accuracy: {best_rbf_acc} (params: {best_rbf_params})\n")

Logistic Regression Accuracy: 0.98
Testing with paramaters: Cs=[0.01, 0.1, 1, 10, 100], Gammas=[0.01, 0.1, 1, 10], Degrees=[2, 3, 4]



For the logistic regression it resulted in an accuracy of $98\%$

The code submitted tests a variety of different kernels (linear, polynomial and RBF/Gaussian) with different regularization paramaters $(0.01,0.1,1,10,100)$, different gamma values $(0.01,0.1,1,10)$ and degree values $(2,3,4)$ for the kernels. Slight caveat that gamma$=10$ and $C=100$ together takes too long and was skipped. This resulted in the best model being an RBF/Gaussian kernel with a regularization parameter of $100$ and a gamma value of $0.01$ which achieved an accuracy of $99.69\%$ on the test set. With the best polynomial kernel being a degree $3$ polynomial with a regularization parameter of $11$ and a gamma value of $0.1$ which achieved an accuracy of $99.45\%$ on the test set. The best linear kernel was with a regularization parameter of $1$ which achieved an accuracy of $99.48\%$ on the test set.